| API | 核心关注点 | 返回类型 | 典型用途 |
| :--- | :--- | :--- | :--- |
| `ray.nodes()` | 节点个体 | 列表 (List) | 检查节点存活状态、IP 地址、单节点资源。 |
| `ray.cluster_resources()` | 集群整体 | 字典 (Dict) | 监控剩余算力、决定是否可以提交新任务。 |
| `ray.timeline()` | 时间序列 | 文件 (JSON) | 深度性能分析、可视化任务调度延迟、查找卡顿原因。 |


这三个 API 分别对应了 Ray 集群管理的三个核心维度：节点状态监控、资源总量统计以及性能深度分析。它们能帮助你从宏观到微观全面掌握集群的运行情况。
以下是详细介绍：
##  ray.nodes()：集群节点的健康检查

这个函数用于获取集群中所有节点的详细状态列表。它是判断集群是否健康、节点是否掉线或资源是否饱和的最直接工具。
- 功能：返回一个字典列表，每个字典代表一个节点（包括 Head 节点和 Worker 节点）的元数据。
- 返回内容：包含节点 IP、端口、状态（alive 或 dead）、该节点拥有的资源（CPU/GPU 数量）以及该节点上运行的 Worker 进程信息等。

In [ ]:
import ray
import json

ray.init()

# 获取所有节点信息
nodes = ray.nodes()

for node in nodes:
    # 判断节点是否存活
    is_alive = node["alive"]
    # 获取节点 IP
    node_ip = node["NodeManagerAddress"]
    # 获取该节点的资源配置
    resources = node["Resources"]
    
    print(f"节点 IP: {node_ip}, 状态: {'存活' if is_alive else '宕机'}, 资源: {resources}")

### 应用场景：
- 故障排查：检查是否有 Worker 节点意外断开（alive: False）。
- 资源分布查看：确认任务是否被调度到了预期的节点上（例如特定的 GPU 节点）

## ray.cluster_resources()：全局资源视图

这个函数用于获取整个集群的总资源量和当前可用资源量。它不关心资源分布在哪个具体节点，而是提供一个聚合的上帝视角。

- 功能：返回一个字典，键是资源名称（如 'CPU', 'GPU', 'memory' 或自定义资源），值是该资源的总量。
- 关键区别：
    - 总资源：集群启动时声明的所有资源之和。
    - 可用资源：当前未被任务占用，可以立即分配的资源。

In [ ]:
import ray

ray.init()

# 获取集群资源总量
total_resources = ray.cluster_resources()

# 获取当前可用资源（动态变化）
available_resources = ray.available_resources()

print(f"集群总资源: {total_resources}")
print(f"当前可用资源: {available_resources}")

# 常见用法：检查是否有 GPU 可用
if available_resources.get("GPU", 0) > 0:
    print("发现空闲 GPU，可以提交推理任务")

### 应用场景：
- 动态调度：<font color='red'>在提交任务前，先检查集群是否有足够的空闲资源，避免任务长时间排队。</font>
- 弹性伸缩判断：<font color='green'>如果 available_resources 长期接近 0，说明集群负载已满，可能需要增加节点。</font>

## ray.timeline(filename)：性能分析的“黑匣子”

这是一个用于性能调优的高级工具。<font color='green'>它会将 Ray 集群中发生的所有事件（任务调度、Actor 创建、对象传输、依赖等待等）记录到一个 JSON 文件中。</font>

- 功能：生成一个符合 Chrome Trace Viewer 格式的 JSON 文件。
- 使用方式：
    1. 在代码中调用 ray.timeline("filename.json")。
    2. 运行脚本，脚本结束后会生成文件。
    3. 在 Chrome 浏览器地址栏输入 chrome://tracing/，点击 "Load" 加载该 JSON 文件。

In [ ]:
import ray
import time

ray.init()

# 开启时间线记录，文件名为 "ray_timeline.json"
ray.timeline("ray_timeline.json")

@ray.remote
def heavy_task():
    time.sleep(1)
    return "Done"

# 提交一些任务
futures = [heavy_task.remote() for _ in range(10)]
ray.get(futures)

# 注意：通常建议在脚本结束时或需要分析的时间点生成文件
# 在某些版本中，可能需要显式停止或等待记录完成

### 可视化分析内容：
在 Chrome Tracing 界面中，你可以看到：
- 时间轴：横轴是时间，纵轴是不同的 Worker 线程。
- 任务耗时：每个矩形代表一个任务，长度代表执行时间。
- 调度延迟：任务从提交到实际开始执行的时间差。
- 依赖等待：任务等待数据（ObjectRef）就绪的时间（通常显示为灰色或特定颜色）。
### 应用场景：
- 定位瓶颈：<font color='green'>区分是计算慢（矩形长）还是调度慢/网络传输慢（矩形前的空白长）。</font>
- 优化并行度：查看任务是否真正在并行执行，还是被串行化了。